# CreditLens — reproduce the evaluation results in Colab

Runs on a free CPU runtime. It clones the public repository, installs the pinned dependencies and

1. recomputes every headline number, table and figure from the committed experiment results (~1 min);
2. optionally re-runs the walk-forward, economic-value and robustness analyses from the frozen panel (~10 min);
3. optionally re-trains the deployed network and baselines and checks they match the committed ones (~5–10 min).

Repository: https://github.com/fabu153/proj · App: https://proj-two-bay.vercel.app

In [ ]:
!git clone --depth 1 https://github.com/fabu153/proj.git creditlens
%cd creditlens
# exact versions used for the committed results; Colab's Python may need the looser requirements-ml.txt instead
!pip install -q -r requirements-repro.txt --extra-index-url https://download.pytorch.org/whl/cpu || pip install -q -r requirements-ml.txt torch --extra-index-url https://download.pytorch.org/whl/cpu

## 1. Headline numbers, tables and figures

In [ ]:
!python -m training.ml.results_summary
import json
n = json.load(open("training/experiments/results_summary_v1/numbers.json"))
print({k: n[k] for k in ["ft_mlp_pr", "ft_logistic_pr", "ft_xgboost_pr", "ft_altman_z2_pr", "wf_mlp_minus_logistic", "ev10_mlp_caught", "ev10_mlp_value"]})

In [ ]:
from IPython.display import Image, display, Markdown
R = "training/experiments/results_summary_v1"
for f in ["fig2_walk_forward", "fig3_capture", "fig4_robustness"]:
    display(Image(f"{R}/figures/{f}.png", width=900))
for t in ["frozen", "walk_forward", "economic", "robustness"]:
    display(Markdown(open(f"{R}/tables/{t}.md").read()))

## 2. Re-run the evaluations from the frozen panel (optional, ~10 min)

In [ ]:
!python -m training.ml.classic_scores --offline
!python -m training.ml.walk_forward
!python -m training.ml.economic_value
!python -m training.ml.robustness
!python -m training.ml.onset_watchlist
!python -m training.ml.supplementary_checks
!python -m training.ml.results_summary
!git diff --stat -- training/experiments   # empty on the same library versions

## 3. Re-train the deployed model and check it matches (optional, ~5–10 min)

In [ ]:
!python -m scripts.verify_training